# Long-Run Ablation Suite

Use this notebook for longer-running follow-up experiments after the main Stage 2 training line.

This notebook covers two experiment families:

- learning-rate ablations at fixed `rank=16` and `epoch=5`
- structure-first then semantics-focused two-stage training

This version is a batch runner. It will execute every preset listed in `RUN_PRESETS` in sequence without manual editing.

Recommended default order:

1. `lr1e4_epoch5_rank16_full`
2. `lr2e4_epoch5_rank16_full`
3. `lr4e4_epoch5_rank16_full`
4. `structure_then_semantics_v1`

Set `SKIP_COMPLETED = True` to resume safely after an interruption. Each preset trains, exports test predictions, evaluates raw predictions, applies repair, and writes repaired evaluation outputs in the same format as the Stage 2 runner.

In [ ]:
# Uncomment this once in a fresh online environment if needed.
# %pip install -q transformers datasets peft accelerate trl bitsandbytes pyyaml jsonschema

In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import gc
import json
import random
import sys

import torch


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'configs').exists() and (candidate / 'src').exists():
            return candidate
    raise RuntimeError('Could not find project root')


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('project_root =', PROJECT_ROOT)
print('python =', sys.version)
print('cuda_available =', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device =', torch.cuda.get_device_name(0))
    print('bf16_supported =', torch.cuda.is_bf16_supported())

from datasets import load_dataset
from peft import LoraConfig, TaskType
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from trl import SFTTrainer

from src.data.io import dump_jsonl, load_jsonl
from src.evaluation.field_analysis import analyze_field_errors
from src.evaluation.metrics import evaluate_sample, summarize_results, try_parse_prediction_text
from src.evaluation.reporting import group_sample_results, write_json_report
from src.inference.repair import repair_prediction
from src.schemas.registry import get_schema
from src.training.formatters import DEFAULT_SYSTEM_PROMPT, build_user_prompt, convert_to_sft_records

In [ ]:
PRESETS = {
    'lr1e4_epoch5_rank16_full': {
        'experiment_name': 'qwen25_3b_stage2_lr1e4_epoch5_rank16_full',
        'mode': 'full',
        'base_model': 'Qwen/Qwen2.5-3B-Instruct',
        'rank': 16,
        'alpha': 32,
        'dropout': 0.05,
        'learning_rate': 1e-4,
        'epochs': 5,
        'batch_size': 8,
        'grad_accum': 4,
        'max_seq_length': 2048,
        'seed': 42,
        'schema_name': 'ticket_schema_v1_reduced',
        'include_schema_definition': False,
    },
    'lr2e4_epoch5_rank16_full': {
        'experiment_name': 'qwen25_3b_stage2_lr2e4_epoch5_rank16_full',
        'mode': 'full',
        'base_model': 'Qwen/Qwen2.5-3B-Instruct',
        'rank': 16,
        'alpha': 32,
        'dropout': 0.05,
        'learning_rate': 2e-4,
        'epochs': 5,
        'batch_size': 8,
        'grad_accum': 4,
        'max_seq_length': 2048,
        'seed': 42,
        'schema_name': 'ticket_schema_v1_reduced',
        'include_schema_definition': False,
    },
    'lr4e4_epoch5_rank16_full': {
        'experiment_name': 'qwen25_3b_stage2_lr4e4_epoch5_rank16_full',
        'mode': 'full',
        'base_model': 'Qwen/Qwen2.5-3B-Instruct',
        'rank': 16,
        'alpha': 32,
        'dropout': 0.05,
        'learning_rate': 4e-4,
        'epochs': 5,
        'batch_size': 8,
        'grad_accum': 4,
        'max_seq_length': 2048,
        'seed': 42,
        'schema_name': 'ticket_schema_v1_reduced',
        'include_schema_definition': False,
    },
    'structure_then_semantics_v1': {
        'experiment_name': 'qwen25_3b_stage2_structure_then_semantics_v1',
        'mode': 'two_stage_structure_semantics',
        'base_model': 'Qwen/Qwen2.5-3B-Instruct',
        'rank': 16,
        'alpha': 32,
        'dropout': 0.05,
        'learning_rate': 2e-4,
        'stage1_epochs': 1,
        'stage2_epochs': 5,
        'batch_size': 8,
        'grad_accum': 4,
        'max_seq_length': 2048,
        'seed': 42,
        'schema_name': 'ticket_schema_v1_reduced',
        'include_schema_definition': False,
        'structure_stage_include_schema_definition': True,
        'structure_stage_buckets': ['simple', 'medium'],
    },
}

RUN_PRESETS = [
    'lr1e4_epoch5_rank16_full',
    'lr2e4_epoch5_rank16_full',
    'lr4e4_epoch5_rank16_full',
    'structure_then_semantics_v1',
]
SKIP_COMPLETED = True

run_configs = [(name, PRESETS[name]) for name in RUN_PRESETS]
print('scheduled_presets =', RUN_PRESETS)
print('skip_completed =', SKIP_COMPLETED)
run_configs

In [ ]:
ARTIFACT_DIR = PROJECT_ROOT / 'data' / 'stage2_long_run'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

train_records = load_jsonl(PROJECT_ROOT / 'data' / 'reduced' / 'phase1_train_reduced.jsonl')
val_records = load_jsonl(PROJECT_ROOT / 'data' / 'reduced' / 'phase1_val_reduced.jsonl')
test_records = load_jsonl(PROJECT_ROOT / 'data' / 'reduced' / 'phase1_test_reduced.jsonl')

def bucket_counts(records):
    counts = defaultdict(int)
    for record in records:
        counts[record['complexity_bucket']] += 1
    return dict(sorted(counts.items()))


def write_sft_split(records, file_name, include_schema_definition=False):
    path = ARTIFACT_DIR / file_name
    dump_jsonl(path, convert_to_sft_records(records, include_schema_definition=include_schema_definition))
    return path


def build_output_paths(experiment_name):
    return {
        'checkpoint_dir': PROJECT_ROOT / 'results' / 'checkpoints' / experiment_name,
        'prediction_path': PROJECT_ROOT / 'results' / 'predictions' / f'{experiment_name}_test.jsonl',
        'repaired_prediction_path': PROJECT_ROOT / 'results' / 'predictions' / f'{experiment_name}_test_repaired.jsonl',
        'raw_report_path': PROJECT_ROOT / 'results' / 'metrics' / f'{experiment_name}_test_report.json',
        'repaired_report_path': PROJECT_ROOT / 'results' / 'metrics' / f'{experiment_name}_test_repaired_report.json',
        'raw_field_path': PROJECT_ROOT / 'results' / 'metrics' / f'{experiment_name}_field_analysis.json',
        'repaired_field_path': PROJECT_ROOT / 'results' / 'metrics' / f'{experiment_name}_test_repaired_field_analysis.json',
    }


def prepare_experiment_artifacts(config):
    experiment_name = config['experiment_name']
    val_path = write_sft_split(val_records, f'{experiment_name}_val.jsonl', include_schema_definition=False)
    artifacts = {
        'experiment_name': experiment_name,
        'val_path': val_path,
    }
    if config['mode'] == 'two_stage_structure_semantics':
        structure_stage_records = [
            record
            for record in train_records
            if record['complexity_bucket'] in set(config['structure_stage_buckets'])
        ]
        semantic_stage_records = list(train_records)
        artifacts['stage1_train_path'] = write_sft_split(
            structure_stage_records,
            f'{experiment_name}_stage1_structure_train.jsonl',
            include_schema_definition=config['structure_stage_include_schema_definition'],
        )
        artifacts['stage2_train_path'] = write_sft_split(
            semantic_stage_records,
            f'{experiment_name}_stage2_semantics_train.jsonl',
            include_schema_definition=False,
        )
        artifacts['structure_bucket_counts'] = bucket_counts(structure_stage_records)
        artifacts['semantic_bucket_counts'] = bucket_counts(semantic_stage_records)
    else:
        artifacts['train_path'] = write_sft_split(
            train_records,
            f'{experiment_name}_train.jsonl',
            include_schema_definition=False,
        )
        artifacts['train_bucket_counts'] = bucket_counts(train_records)
    return artifacts


print('train =', len(train_records))
print('val =', len(val_records))
print('test =', len(test_records))
print('train_bucket_counts =', bucket_counts(train_records))

In [ ]:
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
)


def load_tokenizer(base_model_name):
    tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = 'right'
    return tokenizer


def load_chat_dataset(train_file, validation_file, tokenizer):
    dataset = load_dataset(
        'json',
        data_files={
            'train': str(train_file),
            'validation': str(validation_file),
        },
    )

    def format_chat_example(example):
        example['text'] = tokenizer.apply_chat_template(
            example['messages'],
            tokenize=False,
            add_generation_prompt=False,
        )
        return example

    return dataset.map(format_chat_example)


def load_base_model(base_model_name):
    model = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        quantization_config=bnb_config,
        device_map='auto',
        trust_remote_code=True,
    )
    model.config.use_cache = False
    return model


def build_training_args(config, output_dir, num_epochs):
    return TrainingArguments(
        output_dir=str(output_dir),
        learning_rate=float(config['learning_rate']),
        num_train_epochs=float(num_epochs),
        per_device_train_batch_size=int(config['batch_size']),
        per_device_eval_batch_size=int(config['batch_size']),
        gradient_accumulation_steps=int(config['grad_accum']),
        warmup_steps=50,
        logging_steps=10,
        eval_strategy='steps',
        eval_steps=50,
        save_steps=100,
        save_total_limit=2,
        bf16=use_bf16,
        fp16=not use_bf16,
        report_to='none',
        remove_unused_columns=False,
        seed=int(config['seed']),
    )


def build_peft_config(config):
    return LoraConfig(
        r=int(config['rank']),
        lora_alpha=int(config['alpha']),
        lora_dropout=float(config['dropout']),
        bias='none',
        task_type=TaskType.CAUSAL_LM,
        target_modules='all-linear',
    )


def build_trainer(model, dataset, tokenizer, config, output_dir, num_epochs, peft_config=None):
    training_args = build_training_args(config=config, output_dir=output_dir, num_epochs=num_epochs)
    trainer_kwargs = {
        'model': model,
        'args': training_args,
        'train_dataset': dataset['train'],
        'eval_dataset': dataset['validation'],
        'processing_class': tokenizer,
    }
    if peft_config is not None:
        trainer_kwargs['peft_config'] = peft_config
    return SFTTrainer(**trainer_kwargs)


def build_inference_messages(record):
    return [
        {'role': 'system', 'content': DEFAULT_SYSTEM_PROMPT},
        {
            'role': 'user',
            'content': build_user_prompt(
                input_text=record['input_text'],
                task_name=record['task_name'],
                schema_name=record['schema_name'],
                include_schema_definition=False,
            ),
        },
    ]


def generate_prediction_text(model, tokenizer, record, generation_kwargs):
    messages = build_inference_messages(record)
    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt_text, return_tensors='pt').to(model.device)
    outputs = model.generate(**inputs, **generation_kwargs)
    generated_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()


def sample_eval_dicts(gold_records, pred_records, schema):
    predictions_by_id = {record['sample_id']: record for record in pred_records}
    results = []
    for gold_record in gold_records:
        pred_record = predictions_by_id.get(gold_record['sample_id'], {})
        sample_eval = evaluate_sample(
            sample_id=gold_record['sample_id'],
            prediction_text=pred_record.get('prediction_text'),
            prediction_json=pred_record.get('prediction_json'),
            target_json=gold_record['target_json'],
            schema=schema,
        )
        results.append(
            {
                **sample_eval.__dict__,
                'schema_name': gold_record['schema_name'],
                'complexity_bucket': gold_record.get('complexity_bucket', 'unknown'),
            }
        )
    return results


def summarize_from_dicts(sample_results):
    from src.evaluation.metrics import SampleEvaluation
    evaluations = [
        SampleEvaluation(
            sample_id=item['sample_id'],
            valid_json=item['valid_json'],
            schema_compliant=item['schema_compliant'],
            field_exact_match=item['field_exact_match'],
            exact_match=item['exact_match'],
            primary_error=item['primary_error'],
        )
        for item in sample_results
    ]
    return summarize_results(evaluations)

In [ ]:
generation_kwargs = {
    'max_new_tokens': 256,
    'do_sample': False,
    'temperature': 1.0,
    'top_p': 1.0,
}


def outputs_complete(paths):
    required = [
        paths['prediction_path'],
        paths['repaired_prediction_path'],
        paths['raw_report_path'],
        paths['repaired_report_path'],
        paths['raw_field_path'],
        paths['repaired_field_path'],
    ]
    return all(path.exists() for path in required)


def load_existing_summary(preset_name, config, paths):
    raw_report = json.loads(paths['raw_report_path'].read_text(encoding='utf-8'))
    repaired_report = json.loads(paths['repaired_report_path'].read_text(encoding='utf-8'))
    return {
        'preset_name': preset_name,
        'experiment_name': config['experiment_name'],
        'mode': config['mode'],
        'status': 'skipped_existing',
        'raw_summary': raw_report['summary'],
        'repaired_summary': repaired_report['summary'],
    }


def run_single_experiment(preset_name, config):
    experiment_name = config['experiment_name']
    paths = build_output_paths(experiment_name)
    if SKIP_COMPLETED and outputs_complete(paths):
        print('skipping completed preset =', preset_name)
        return load_existing_summary(preset_name, config, paths)

    artifacts = prepare_experiment_artifacts(config)
    schema = get_schema(config['schema_name'])
    base_model_name = config['base_model']
    tokenizer = load_tokenizer(base_model_name)
    model = load_base_model(base_model_name)
    output_root = paths['checkpoint_dir']
    output_root.mkdir(parents=True, exist_ok=True)
    print('pad_token =', tokenizer.pad_token)
    print('eos_token =', tokenizer.eos_token)

    trainers_to_cleanup = []
    try:
        if config['mode'] == 'two_stage_structure_semantics':
            print('stage1 structure buckets =', artifacts['structure_bucket_counts'])
            print('stage2 semantic buckets =', artifacts['semantic_bucket_counts'])
            stage1_dataset = load_chat_dataset(artifacts['stage1_train_path'], artifacts['val_path'], tokenizer)
            stage1_output = output_root / 'stage1_structure'
            stage1_trainer = build_trainer(
                model=model,
                dataset=stage1_dataset,
                tokenizer=tokenizer,
                config=config,
                output_dir=stage1_output,
                num_epochs=config['stage1_epochs'],
                peft_config=build_peft_config(config),
            )
            trainers_to_cleanup.append(stage1_trainer)
            stage1_result = stage1_trainer.train()
            stage1_trainer.save_model(str(stage1_output))
            model = stage1_trainer.model
            print('stage1_train_loss =', stage1_result.training_loss)
            print('stage1 structure-focused phase finished')

            stage2_dataset = load_chat_dataset(artifacts['stage2_train_path'], artifacts['val_path'], tokenizer)
            stage2_output = output_root / 'stage2_semantics'
            stage2_trainer = build_trainer(
                model=model,
                dataset=stage2_dataset,
                tokenizer=tokenizer,
                config=config,
                output_dir=stage2_output,
                num_epochs=config['stage2_epochs'],
                peft_config=None,
            )
            trainers_to_cleanup.append(stage2_trainer)
            stage2_result = stage2_trainer.train()
            stage2_trainer.save_model(str(output_root))
            print('stage2_train_loss =', stage2_result.training_loss)
        else:
            print('train buckets =', artifacts['train_bucket_counts'])
            dataset = load_chat_dataset(artifacts['train_path'], artifacts['val_path'], tokenizer)
            trainer = build_trainer(
                model=model,
                dataset=dataset,
                tokenizer=tokenizer,
                config=config,
                output_dir=output_root,
                num_epochs=config['epochs'],
                peft_config=build_peft_config(config),
            )
            trainers_to_cleanup.append(trainer)
            train_result = trainer.train()
            trainer.save_model(str(output_root))
            print('train_loss =', train_result.training_loss)

        print('saved_checkpoint_dir =', output_root)

        predictions = []
        for idx, record in enumerate(test_records):
            prediction_text = generate_prediction_text(model, tokenizer, record, generation_kwargs)
            try:
                prediction_json = json.loads(prediction_text)
            except json.JSONDecodeError:
                prediction_json = None
            predictions.append(
                {
                    'sample_id': record['sample_id'],
                    'prediction_text': prediction_text,
                    'prediction_json': prediction_json,
                    'metadata': {
                        'model_name': base_model_name,
                        'experiment_id': experiment_name,
                    },
                }
            )
            if (idx + 1) % 25 == 0:
                print(f'generated {idx + 1} / {len(test_records)}')

        dump_jsonl(paths['prediction_path'], predictions)

        raw_sample_results = sample_eval_dicts(test_records, predictions, schema)
        raw_report = {
            'summary': summarize_from_dicts(raw_sample_results),
            'grouped_summary': {
                'by_complexity_bucket': {
                    name: summarize_from_dicts(items)
                    for name, items in group_sample_results(raw_sample_results, 'complexity_bucket').items()
                },
            },
            'per_sample': raw_sample_results,
        }
        write_json_report(paths['raw_report_path'], raw_report)
        write_json_report(paths['raw_field_path'], analyze_field_errors(test_records, predictions))

        repaired_predictions = []
        for record in predictions:
            prediction_json = record.get('prediction_json')
            prediction_text = record.get('prediction_text')
            if prediction_json is None and isinstance(prediction_text, str):
                _, prediction_json = try_parse_prediction_text(prediction_text)
            repaired_json, repaired = repair_prediction(prediction_json, schema)
            repaired_predictions.append(
                {
                    **record,
                    'prediction_json': repaired_json,
                    'metadata': {
                        **record.get('metadata', {}),
                        'repaired': repaired,
                    },
                }
            )

        repaired_sample_results = sample_eval_dicts(test_records, repaired_predictions, schema)
        repaired_report = {
            'summary': summarize_from_dicts(repaired_sample_results),
            'per_sample': repaired_sample_results,
        }
        dump_jsonl(paths['repaired_prediction_path'], repaired_predictions)
        write_json_report(paths['repaired_report_path'], repaired_report)
        write_json_report(paths['repaired_field_path'], analyze_field_errors(test_records, repaired_predictions))
        print('raw_report =', paths['raw_report_path'])
        print('repaired_report =', paths['repaired_report_path'])

        return {
            'preset_name': preset_name,
            'experiment_name': experiment_name,
            'mode': config['mode'],
            'status': 'completed',
            'raw_summary': raw_report['summary'],
            'repaired_summary': repaired_report['summary'],
        }
    finally:
        for trainer in trainers_to_cleanup:
            del trainer
        del model
        del tokenizer
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


batch_run_results = []
for preset_name, config in run_configs:
    print('\n' + '=' * 80)
    print('running preset =', preset_name)
    print('=' * 80)
    result = run_single_experiment(preset_name, config)
    batch_run_results.append(result)
    print('raw_summary =', json.dumps(result['raw_summary'], indent=2))
    print('repaired_summary =', json.dumps(result['repaired_summary'], indent=2))

batch_run_results

In [ ]:
batch_summary_path = PROJECT_ROOT / 'docs' / 'results' / 'long_run_ablation_batch_summary.md'
lines = [
    '# Long-Run Ablation Batch Summary',
    '',
    f'Skip completed: `{SKIP_COMPLETED}`',
    '',
    '## Runs',
    '',
]
for item in batch_run_results:
    raw = item['raw_summary']
    repaired = item['repaired_summary']
    lines.extend([
        f"### {item['preset_name']}",
        '',
        f"- experiment: `{item['experiment_name']}`",
        f"- mode: `{item['mode']}`",
        f"- status: `{item['status']}`",
        f"- raw field exact match: `{raw['field_exact_match']:.4f}`",
        f"- raw end-to-end exact match: `{raw['end_to_end_exact_match']:.4f}`",
        f"- repaired field exact match: `{repaired['field_exact_match']:.4f}`",
        f"- repaired end-to-end exact match: `{repaired['end_to_end_exact_match']:.4f}`",
        '',
    ])

batch_summary_path.write_text('\n'.join(lines) + '\n', encoding='utf-8')
print('batch_summary_path =', batch_summary_path)
print(batch_summary_path.read_text(encoding='utf-8'))

In [ ]:
leaderboard = sorted(
    [
        {
            'preset_name': item['preset_name'],
            'experiment_name': item['experiment_name'],
            'status': item['status'],
            'field_exact_match': item['raw_summary']['field_exact_match'],
            'end_to_end_exact_match': item['raw_summary']['end_to_end_exact_match'],
        }
        for item in batch_run_results
    ],
    key=lambda row: row['end_to_end_exact_match'],
    reverse=True,
)
leaderboard

In [ ]:
for row in leaderboard:
    print(row)